In [ ]:
!pip install scikeras

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
import warnings
warnings.filterwarnings('ignore')

# For data preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.metrics import (classification_report, roc_curve, auc, accuracy_score,
                            precision_recall_curve, confusion_matrix, average_precision_score,
                            f1_score, precision_score, recall_score)
from sklearn.feature_selection import RFECV, mutual_info_classif
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

# For imbalanced data handling
from imblearn.over_sampling import SMOTE

# Deep Learning libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                    Activation, Input, Reshape, Conv1D,
                                    MaxPooling1D, Flatten, GlobalAveragePooling1D)
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
                                       ReduceLROnPlateau, TensorBoard)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import plot_model
import pickle

# Set display precision & style
np.set_printoptions(precision=4)
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-pastel')

# ---------------------------
# 0. Loading and exploring the dataset
# ---------------------------
print("="*80)
print("PHISHING DETECTION WITH CNN - ADVANCED ANALYSIS")
print("="*80)

# Load the dataset
df = pd.read_csv("Phishing_Legitimate_full.csv")

print(f"Dataset dimensions: {df.shape[0]} rows × {df.shape[1]} columns")

# Create a figure with two subplots for initial exploration
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Display class distribution
target_column = 'CLASS_LABEL' if 'CLASS_LABEL' in df.columns else 'label'
class_counts = df[target_column].value_counts()
colors = ['#6495ED', '#FF7F50']
ax1.pie(class_counts, labels=class_counts.index, autopct='%1.1f%%',
        colors=colors, shadow=True, startangle=90, explode=(0.1, 0))
ax1.set_title('Class Distribution', fontsize=14, pad=20)

# Display descriptive statistics for a few numeric columns
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns[:10]
stats_df = df[numeric_columns].describe().T[['mean', 'std', 'min', 'max']]
sns.heatmap(stats_df, cmap='coolwarm', annot=True, fmt='.2f', ax=ax2)
ax2.set_title('Descriptive Statistics (First 10 variables)', fontsize=14, pad=20)

plt.tight_layout()
plt.savefig('class_distribution.png')
plt.close()

# ---------------------------
# 1. Advanced Data Preparation
# ---------------------------
print("\n" + "="*80)
print("DATA PREPARATION AND PREPROCESSING")
print("="*80)

# Remove non-relevant columns (e.g., 'id' and 'url')
for col in ['id', 'url']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# Check and handle missing values
missing_values = df.isnull().sum()
print("\nMissing values per column:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values")

# Remove observations with missing values in the target (if necessary)
df.dropna(subset=[target_column], inplace=True)

# Check correlations between variables
corr_matrix = df.corr()
plt.figure(figsize=(16, 12))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            square=True, linewidths=.5, annot=False, cbar_kws={'shrink': .5})
plt.title('Correlation Matrix of Variables', fontsize=16, pad=20)
plt.tight_layout()
plt.savefig('correlation_matrix.png')
plt.close()

# Separate features and target
X = df.drop(columns=[target_column])
y = df[target_column]
feature_names = X.columns

# Analysis of initial features
print(f"\nNumber of initial features: {X.shape[1]}")
print(f"Initial class distribution: {y.value_counts().to_dict()}")

# ---------------------------
# 2. Advanced preprocessing: Scaling with PowerTransformer + StandardScaler
# ---------------------------
# PowerTransformer to normalize distributions
print("\nApplying transformations to normalize distributions...")
power_transformer = PowerTransformer(method='yeo-johnson')
X_transformed = power_transformer.fit_transform(X)

# Standard Scaler to center and reduce
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_transformed)
print("Transformations applied.")

# ---------------------------
# 3. Anomaly detection and removal with IsolationForest
# ---------------------------
print("\nAnomaly detection with IsolationForest:")
chosen_contamination = 0.01
iso = IsolationForest(contamination=chosen_contamination, random_state=42, n_jobs=-1)
pred_anomalies = iso.fit_predict(X_scaled)
mask = pred_anomalies != -1
X_clean = X_scaled[mask]
y_clean = y[mask]

# Visualize anomalies (PCA for 2D visualization)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=y[mask], cmap='viridis', alpha=0.6, s=30, edgecolor='k', linewidth=0.3)
plt.scatter(X_pca[~mask, 0], X_pca[~mask, 1], c='red', marker='x', s=50, label='Anomalies')
plt.title('Visualization of Detected Anomalies (PCA)', fontsize=14)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.tight_layout()
plt.savefig('anomalies_pca.png')
plt.close()

anomaly_count = len(X_scaled) - np.sum(mask)
print(f"Points removed as anomalies: {anomaly_count} ({anomaly_count/len(X_scaled)*100:.2f}%)")

# ---------------------------
# 4. Class balancing with SMOTE
# ---------------------------
print("\nBalancing classes with SMOTE...")
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_clean, y_clean)

# Visualization before/after balancing
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Before SMOTE
ax1.bar(['Legitimate', 'Phishing'], [np.sum(y_clean == 0), np.sum(y_clean == 1)],
        color=['#6495ED', '#FF7F50'])
ax1.set_title('Class Distribution - Before SMOTE', fontsize=14)
ax1.set_ylabel('Number of samples')

# After SMOTE
ax2.bar(['Legitimate', 'Phishing'], [np.sum(y_res == 0), np.sum(y_res == 1)],
        color=['#6495ED', '#FF7F50'])
ax2.set_title('Class Distribution - After SMOTE', fontsize=14)
ax2.set_ylabel('Number of samples')

plt.tight_layout()
plt.savefig('smote_balancing.png')
plt.close()

print(f"Class distribution after SMOTE: {np.bincount(y_res)}")

# ---------------------------
# 5. RFECV for optimal feature selection
# ---------------------------
print("\n" + "="*80)
print("FEATURE SELECTION WITH RFECV")
print("="*80)

# Use a simple model as the basis for RFECV
from sklearn.ensemble import RandomForestClassifier
base_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# RFECV configuration for optimal feature selection
rfecv = RFECV(
    estimator=base_clf,
    step=1,  # Eliminate 1 feature at each step
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',  # F1 score as metric (well adapted to imbalanced problems)
    n_jobs=-1,  # Use all available processors
    verbose=1,
    min_features_to_select=5  # Keep at least 5 features
)

print("\nRunning feature selection with RFECV (please be patient, this may take time)...")
rfecv.fit(X_res, y_res)

# RFECV results
print(f"\nOptimal number of features: {rfecv.n_features_}")
if rfecv.n_features_ <= len(rfecv.cv_results_['mean_test_score']):
    print(f"Score with {rfecv.n_features_} features: {rfecv.cv_results_['mean_test_score'][rfecv.n_features_ - 1]:.4f}")
else:
    last_idx = len(rfecv.cv_results_['mean_test_score']) - 1
    print(f"Score with {last_idx + 1} features: {rfecv.cv_results_['mean_test_score'][last_idx]:.4f} (adjusted)")
# RFECV scores visualization
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1), rfecv.cv_results_['mean_test_score'], marker='o',
         color='#6495ED', linestyle='-', linewidth=2, markersize=8)
plt.axvline(x=rfecv.n_features_, color='#FF7F50', linestyle='--', label=f'Optimal: {rfecv.n_features_} features')
plt.fill_between(range(1, len(rfecv.cv_results_['mean_test_score']) + 1), rfecv.cv_results_['mean_test_score'] - np.std(rfecv.cv_results_['mean_test_score']),
                rfecv.cv_results_['mean_test_score'] + np.std(rfecv.cv_results_['mean_test_score']), alpha=0.2, color='#6495ED')
plt.xlabel('Number of Features', fontsize=12)
plt.ylabel('Cross-Validation Score (F1)', fontsize=12)
plt.title('Cross-Validation Scores vs Number of Features', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.savefig('rfecv_scores.png')
plt.close()

# Get selected features
selected_features_mask = rfecv.support_
selected_feature_names = feature_names[selected_features_mask]

# Display selected features
print("\nFeatures selected by RFECV:")
for i, feature in enumerate(selected_feature_names):
    print(f"{i+1}. {feature}")

# Transform the data to keep only selected features
X_selected = rfecv.transform(X_res)

# ---------------------------
# 6. Feature importance analysis
# ---------------------------
# Calculate feature importance via mutual information
mi_scores = mutual_info_classif(X_selected, y_res, random_state=42)
mi_df = pd.DataFrame({'Feature': selected_feature_names, 'Importance': mi_scores})
mi_df = mi_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=mi_df, palette='viridis')
plt.title('Feature Importance (Mutual Information)', fontsize=14)
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.close()

# ---------------------------
# 7. Train/test split
# ---------------------------
print("\n" + "="*80)
print("CNN MODEL TRAINING AND OPTIMIZATION")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_res, test_size=0.2, random_state=42, stratify=y_res
)

print(f"Training set dimensions: {X_train.shape}")
print(f"Test set dimensions: {X_test.shape}")

# ---------------------------
# 8. CNN Model Definition
# ---------------------------
# Reshape data for CNN (CNN expects 3D input)
# For a 1D CNN, we need to reshape from (samples, features) to (samples, features, 1)
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"CNN input shape: {X_train_cnn.shape}")

# Define the CNN model
def create_cnn_model(input_shape, learning_rate=0.001):
    model = Sequential([
        # First convolutional layer
        Conv1D(filters=64, kernel_size=3, padding='same', activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        # Second convolutional layer
        Conv1D(filters=128, kernel_size=3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        # Third convolutional layer
        Conv1D(filters=256, kernel_size=3, padding='same', activation='relu'),
        BatchNormalization(),
        GlobalAveragePooling1D(),  # Alternative to flatten

        # Dense layers
        Dense(128, activation='relu'),
        Dropout(0.5),
        BatchNormalization(),

        Dense(64, activation='relu'),
        Dropout(0.3),
        BatchNormalization(),

        # Output layer
        Dense(1, activation='sigmoid')
    ])

    # Compile the model
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(),
                 tf.keras.metrics.Recall(),
                 tf.keras.metrics.AUC()]
    )

    return model

# Create the model
input_shape = (X_train_cnn.shape[1], 1)
cnn_model = create_cnn_model(input_shape)

# Model summary
print("\nCNN Model Architecture:")
cnn_model.summary()

# ---------------------------
# 9. Model Training with callbacks
# ---------------------------
# Define callbacks for better training
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=1
    ),
    ModelCheckpoint(
        'best_cnn_phishing_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# Train the model
print("\nTraining the CNN model...")
history = cnn_model.fit(
    X_train_cnn, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# ---------------------------
# 10. Learning Curves Visualization
# ---------------------------
# Plot training history
plt.figure(figsize=(15, 10))

# Accuracy subplot
plt.subplot(2, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Loss subplot
plt.subplot(2, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Precision subplot
plt.subplot(2, 2, 3)
plt.plot(history.history['precision'], label='Training Precision')
plt.plot(history.history['val_precision'], label='Validation Precision')
plt.title('Model Precision', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Recall subplot
plt.subplot(2, 2, 4)
plt.plot(history.history['recall'], label='Training Recall')
plt.plot(history.history['val_recall'], label='Validation Recall')
plt.title('Model Recall', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Recall', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cnn_learning_curves.png')
plt.close()

# ---------------------------
# 11. Model Evaluation
# ---------------------------
print("\n" + "="*80)
print("PERFORMANCE EVALUATION AND VISUALIZATION")
print("="*80)

# Evaluate the model on the test set
test_results = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\nTest Loss: {test_results[0]:.4f}")
print(f"Test Accuracy: {test_results[1]:.4f}")
print(f"Test Precision: {test_results[2]:.4f}")
print(f"Test Recall: {test_results[3]:.4f}")
print(f"Test AUC: {test_results[4]:.4f}")

# Get predictions
y_prob = cnn_model.predict(X_test_cnn).flatten()
y_pred = (y_prob >= 0.5).astype(int)

# Classification report
print("\n--- Classification Report ---")
report = classification_report(y_test, y_pred, digits=4)
print(report)

# Confusion Matrix
plt.figure(figsize=(8, 7))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix', fontsize=14)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('cnn_confusion_matrix.png')
plt.close()

# ROC and PR curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curve', fontsize=14)
ax1.legend(loc="lower right")
ax1.grid(True, alpha=0.3)

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
ax2.plot(recall, precision, color='blue', lw=2, label=f'AP = {pr_auc:.4f}')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curve', fontsize=14)
ax2.legend(loc="lower left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cnn_roc_pr_curves.png')
plt.close()

# ---------------------------
# 12. Error Analysis
# ---------------------------
# Indices where predictions are wrong
error_indices = np.where(y_pred != y_test)[0]

if len(error_indices) > 0:
    print(f"\nAnalysis of the first {min(10, len(error_indices))} errors:")

    for i, idx in enumerate(error_indices[:10]):
        print(f"Example {i+1}:")
        # Convert y_test and y_pred to numpy array if not already done
        y_test_array = np.array(y_test) if not isinstance(y_test, np.ndarray) else y_test
        y_pred_array = np.array(y_pred) if not isinstance(y_pred, np.ndarray) else y_pred

        print(f"  Prediction: {y_pred_array[idx]}, True label: {y_test_array[idx]}")
        print(f"  Predicted probability: {y_prob[idx]:.4f}")
        print()

# ---------------------------
# 13. Feature Visualization with tSNE
# ---------------------------
# Extract features from the last convolutional layer for visualization
feature_extractor = Model(
    inputs=cnn_model.inputs,
    outputs=cnn_model.layers[-4].output  # Assuming the fourth to last layer is the GlobalAveragePooling layer
)

# Extract features
conv_features = feature_extractor.predict(X_test_cnn)

# Apply t-SNE for visualization
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_results = tsne.fit_transform(conv_features)

# Create a scatter plot colored by classes
plt.figure(figsize=(12, 10))
scatter = plt.scatter(tsne_results[:, 0], tsne_results[:, 1],
                      c=y_test, cmap='coolwarm', s=50, alpha=0.6, edgecolor='k', linewidth=0.3)

# Mark prediction errors
plt.scatter(tsne_results[error_indices, 0], tsne_results[error_indices, 1],
            marker='x', s=100, color='black', label='Errors')

plt.title('t-SNE Visualization of CNN Features', fontsize=14)
plt.xlabel('t-SNE Component 1', fontsize=12)
plt.ylabel('t-SNE Component 2', fontsize=12)
plt.legend(*scatter.legend_elements(), title="Classes")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cnn_tsne_features.png')
plt.close()

# ---------------------------
# 14. Model and Result Saving
# ---------------------------
print("\nSaving the CNN model...")
cnn_model.save('cnn_phishing_model.h5')
print("Model saved as 'cnn_phishing_model.h5'")

# Calculate final metrics
final_metrics = {
    'confusion_matrix': confusion_matrix(y_test, y_pred),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'fpr': fpr,
    'tpr': tpr,
    'roc_auc': roc_auc,
    'precision_curve': precision,
    'recall_curve': recall,
    'pr_auc': pr_auc,
    'test_results': test_results
}

# Save model, preprocessors, and metrics
model_data = {
    'power_transformer': power_transformer,
    'scaler': scaler,
    'selected_features_mask': selected_features_mask,
    'selected_feature_names': selected_feature_names,
    'feature_names': feature_names,
    'metrics': final_metrics,
    'input_shape': input_shape
}

with open('cnn_phishing_model_data.pkl', 'wb') as file:
    pickle.dump(model_data, file)

print("\nMetrics and preprocessors saved as 'cnn_phishing_model_data.pkl'")

# Compare CNN with original model metrics
if False:  # Set to True if you have already run the LGBM model and have its metrics
    try:
        with open('lightgbm_phishing_model.pkl', 'rb') as file:
            lgbm_data = pickle.load(file)

        # Create comparison plot
        plt.figure(figsize=(12, 8))

        metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
        cnn_scores = [final_metrics[m] for m in metrics]
        lgbm_scores = [lgbm_data['metrics'][m] for m in metrics]

        x = np.arange(len(metrics))
        width = 0.35

        plt.bar(x - width/2, cnn_scores, width, label='CNN', color='#6495ED')
        plt.bar(x + width/2, lgbm_scores, width, label='LightGBM', color='#FF7F50')

        plt.xlabel('Metric', fontsize=12)
        plt.ylabel('Score', fontsize=12)
        plt.title('CNN vs LightGBM Performance Comparison', fontsize=14)
        plt.xticks(x, metrics)
        plt.ylim(0, 1)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('model_comparison.png')
        plt.close()

        print("\nModel comparison completed and saved.")
    except:
        print("\nCould not load LightGBM model data for comparison.")

print("\n" + "="*80)
print("ANALYSIS COMPLETED")
print("="*80)

PHISHING DETECTION WITH CNN - ADVANCED ANALYSIS
Dataset dimensions: 10000 rows × 50 columns

DATA PREPARATION AND PREPROCESSING

Missing values per column:
No missing values

Number of initial features: 48
Initial class distribution: {1: 5000, 0: 5000}

Applying transformations to normalize distributions...
Transformations applied.

Anomaly detection with IsolationForest:
Points removed as anomalies: 99 (0.99%)

Balancing classes with SMOTE...
Class distribution after SMOTE: [4987 4987]

FEATURE SELECTION WITH RFECV

Running feature selection with RFECV (please be patient, this may take time)...
Fitting estimator with 48 features.

Optimal number of features: 47
Score with 44 features: 0.9830 (adjusted)

Features selected by RFECV:
1. NumDots
2. SubdomainLevel
3. PathLevel
4. UrlLength
5. NumDash
6. NumDashInHostname
7. AtSymbol
8. TildeSymbol
9. NumUnderscore
10. NumPercent
11. NumQueryComponents
12. NumAmpersand
13. NumHash
14. NumNumericChars
15. NoHttps
16. RandomString
17. IpAddre

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 47, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 47, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 23, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 23, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 23, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 11, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 11, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 11, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 167,297 (653.50 KB)

 Trainable params: 166,017 (648.50 KB)

...

100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 49ms/step - accuracy: 0.9911 - auc: 0.9996 - loss: 0.0276 - precision: 0.9924 - recall: 0.9903 - val_accuracy: 0.9825 - val_auc: 0.9974 - val_loss: 0.0543 - val_precision: 0.9831 - val_recall: 0.9831 - learning_rate: 5.0000e-04
Epoch 16/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.9940 - auc: 0.9998 - loss: 0.0184 - precision: 0.9930 - recall: 0.9946
Epoch 16: val_loss did not improve from 0.05433
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.9939 - auc: 0.9998 - loss: 0.0184 - precision: 0.9930 - recall: 0.9946 - val_accuracy: 0.9825 - val_auc: 0.9967 - val_loss: 0.0601 - val_precision: 0.9843 - val_recall: 0.9819 - learning_rate: 5.0000e-04
Epoch 17/50
 99/100 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9909 - auc: 0.9993 - loss: 0.0255 - precision: 0.9893 - recall: 0.9920
Epoch 17: val_loss did not improve from 0.05433
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.9909 - auc: 0.9993 - loss: 0.0255 - precision: 0.9

100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.9875 - auc: 0.9990 - loss: 0.0324 - precision: 0.9843 - recall: 0.9905 - val_accuracy: 0.9862 - val_auc: 0.9960 - val_loss: 0.0512 - val_precision: 0.9844 - val_recall: 0.9891 - learning_rate: 5.0000e-04
Epoch 19/50
 99/100 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.9881 - auc: 0.9989 - loss: 0.0320 - precision: 0.9848 - recall: 0.9909
Epoch 19: val_loss did not improve from 0.05121
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - accuracy: 0.9882 - auc: 0.9989 - loss: 0.0318 - precision: 0.9849 - recall: 0.9909 - val_accuracy: 0.9843 - val_auc: 0.9963 - val_loss: 0.0571 - val_precision: 0.9843 - val_recall: 0.9855 - learning_rate: 5.0000e-04
Epoch 20/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.9896 - auc: 0.9993 - loss: 0.0272 - precision: 0.9884 - recall: 0.9902
Epoch 20: val_loss did not improve from 0.05121
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.9896 - auc: 0.9993 - loss: 0.0271 - precision: 0.9

100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - accuracy: 0.9894 - auc: 0.9992 - loss: 0.0290 - precision: 0.9904 - recall: 0.9877 - val_accuracy: 0.9856 - val_auc: 0.9978 - val_loss: 0.0496 - val_precision: 0.9867 - val_recall: 0.9855 - learning_rate: 5.0000e-04
Epoch 23/50
 99/100 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9945 - auc: 0.9997 - loss: 0.0180 - precision: 0.9934 - recall: 0.9956
Epoch 23: val_loss did not improve from 0.04964
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.9944 - auc: 0.9997 - loss: 0.0180 - precision: 0.9934 - recall: 0.9956 - val_accuracy: 0.9843 - val_auc: 0.9951 - val_loss: 0.0617 - val_precision: 0.9808 - val_recall: 0.9891 - learning_rate: 5.0000e-04
Epoch 24/50
 99/100 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.9917 - auc: 0.9997 - loss: 0.0234 - precision: 0.9940 - recall: 0.9892
Epoch 24: val_loss did not improve from 0.04964
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 56ms/step - accuracy: 0.9917 - auc: 0.9997 - loss: 0.0234 - precision: 0.9

100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.9953 - auc: 0.9999 - loss: 0.0152 - precision: 0.9969 - recall: 0.9938 - val_accuracy: 0.9868 - val_auc: 0.9950 - val_loss: 0.0483 - val_precision: 0.9903 - val_recall: 0.9843 - learning_rate: 2.5000e-04
Epoch 29/50
 99/100 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9976 - auc: 1.0000 - loss: 0.0096 - precision: 0.9975 - recall: 0.9977
Epoch 29: val_loss did not improve from 0.04833
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.9976 - auc: 1.0000 - loss: 0.0097 - precision: 0.9975 - recall: 0.9977 - val_accuracy: 0.9875 - val_auc: 0.9944 - val_loss: 0.0542 - val_precision: 0.9903 - val_recall: 0.9855 - learning_rate: 2.5000e-04
Epoch 30/50
 99/100 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9934 - auc: 0.9999 - loss: 0.0151 - precision: 0.9940 - recall: 0.9930
Epoch 30: val_loss did not improve from 0.04833
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - accuracy: 0.9935 - auc: 0.9999 - loss: 0.0150 - precision: 0.9


Saving the CNN model...
Model saved as 'cnn_phishing_model.h5'

Metrics and preprocessors saved as 'cnn_phishing_model_data.pkl'

ANALYSIS COMPLETED
